# PCL detection: three training setups

Based on `pcl_imbalance_weights_only.ipynb`. RoBERTa-base with HTML stripping. **Three setups:**
1. **Original**: Fixed class-weighted CE loss, default (linear) LR scheduler.
2. **Decreasing class weights**: Dynamic weight schedule (w1 starts at 1.5× base, linearly decreases to base).
3. **Decreasing class weights + LR scheduler**: Same decreasing weight schedule + cosine LR scheduler.

In [ ]:
import html
import json
import random
import re
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, precision_recall_fscore_support
from sklearn.model_selection import train_test_split

from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
    set_seed,
)

SEED = 42
MODEL_NAME = "roberta-base"
MAX_LENGTH = 512
BATCH_SIZE = 32
EVAL_BATCH_SIZE = 32
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
EPOCHS = 10
EARLY_STOPPING_PATIENCE = 3
USE_CLASS_WEIGHTS = True  # class-weighted CE loss only

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
set_seed(SEED)

# Resolve project root regardless of whether notebook is run from project root or src/
cwd = Path.cwd().resolve()
if (cwd / "data").exists():
    PROJECT_ROOT = cwd
elif (cwd.parent / "data").exists():
    PROJECT_ROOT = cwd.parent
else:
    raise FileNotFoundError("Could not locate project root containing data/ directory.")

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"
MODEL_BASE = PROJECT_ROOT / "models" / "roberta_pcl_three_setups"
OUTPUT_DIR = PROJECT_ROOT / "output"
OUTPUT_PREFIX = "roberta_pcl_three_setups"

MODEL_BASE.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
print(f"Project root: {PROJECT_ROOT}")

## Load data and build train/dev splits

In [ ]:
df = pd.read_csv(
    RAW_DATA_DIR / "dontpatronizeme_pcl.tsv",
    sep="\t",
    skiprows=4,
    names=["par_id", "art_id", "keyword", "country_code", "text", "label"],
)

df["label_binary"] = df["label"].apply(lambda x: 0 if x in [0, 1] else 1)

# Split IDs (match organizer: exactly one row per ID from split files)
a_train = pd.read_csv(RAW_DATA_DIR / "train_semeval_parids-labels.csv")
a_dev = pd.read_csv(RAW_DATA_DIR / "dev_semeval_parids-labels.csv")

df_sub = df[["par_id", "text", "label_binary"]].drop_duplicates(subset=["par_id"], keep="first").copy()
df_sub["text"] = df_sub["text"].fillna("").astype(str)

train_df = a_train[["par_id"]].merge(df_sub, on="par_id", how="left")
dev_df = a_dev[["par_id"]].merge(df_sub, on="par_id", how="left")
train_df["text"] = train_df["text"].fillna("").astype(str)
dev_df["text"] = dev_df["text"].fillna("").astype(str)
train_df["label_binary"] = train_df["label_binary"].fillna(0).astype(int)
dev_df["label_binary"] = dev_df["label_binary"].fillna(0).astype(int)

# Split a validation set from training data; dev is held out and not used during training
VAL_RATIO = 0.1
train_df, val_df = train_test_split(
    train_df, test_size=VAL_RATIO, stratify=train_df["label_binary"], random_state=SEED
)
train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)

print(f"Train size: {len(train_df)}")
print(f"Val size:   {len(val_df)}")
print(f"Dev size:   {len(dev_df)} (held out, not used during training)")
print("\nTrain label distribution:")
print(train_df["label_binary"].value_counts())
print("\nVal label distribution:")
print(val_df["label_binary"].value_counts())
print("\nDev label distribution:")
print(dev_df["label_binary"].value_counts())

Train size: 7537
Val size:   838
Dev size:   2094 (held out, not used during training)

Train label distribution:
label_binary
0    6822
1     715
Name: count, dtype: int64

Val label distribution:
label_binary
0    759
1     79
Name: count, dtype: int64

Dev label distribution:
label_binary
0    1895
1     199
Name: count, dtype: int64


## Preprocess text: remove HTML markups

From data exploration: texts can contain HTML tags (e.g. `<h>`, `</p>`) that do not add useful signal. Strip tags and unescape HTML entities, then normalize whitespace.

In [ ]:
def strip_html_and_clean(text: str) -> str:
    """Remove HTML tags, unescape entities, and normalize whitespace."""
    if not isinstance(text, str) or not text.strip():
        return text.strip() if isinstance(text, str) else ""
    # Remove HTML tags (pattern from data_exploration: <[^>]+>)
    no_tags = re.sub(r"<[^>]+>", "", text)
    # Unescape HTML entities (e.g. &amp; -> &, &lt; -> <)
    unescaped = html.unescape(no_tags)
    # Collapse multiple spaces and strip
    return re.sub(r"\s+", " ", unescaped).strip()


train_df["text"] = train_df["text"].apply(strip_html_and_clean)
val_df["text"] = val_df["text"].apply(strip_html_and_clean)
dev_df["text"] = dev_df["text"].apply(strip_html_and_clean)

# Sanity check: ensure no empty texts
train_df["text"] = train_df["text"].replace("", " ")
val_df["text"] = val_df["text"].replace("", " ")
dev_df["text"] = dev_df["text"].replace("", " ")
print("HTML preprocessing applied to train, val, and dev texts.")

HTML preprocessing applied to train, val, and dev texts.


In [ ]:
# Class weights for loss (inverse frequency). base_class_weights used by DynamicWeightTrainer.
n0 = (train_df["label_binary"] == 0).sum()
n1 = (train_df["label_binary"] == 1).sum()
n = n0 + n1
w0 = n / (2 * n0)
w1 = n / (2 * n1)
class_weights = torch.tensor([w0, w1], dtype=torch.float32)
base_class_weights = class_weights  # alias for weight-schedule trainer

print(f"Class weights (0, 1): {class_weights.tolist()}")

Class weights (0, 1): [0.55240398645401, 5.270629405975342]


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class PCLDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tokenizer(
            self.texts[idx],
            truncation=True,
            max_length=self.max_length,
            return_attention_mask=True,
        )
        enc["labels"] = int(self.labels[idx])
        return enc

train_dataset = PCLDataset(train_df["text"], train_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
val_dataset = PCLDataset(val_df["text"], val_df["label_binary"], tokenizer, max_length=MAX_LENGTH)
dev_dataset = PCLDataset(dev_df["text"], dev_df["label_binary"], tokenizer, max_length=MAX_LENGTH)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

print("Tokenizer and datasets ready.")

Tokenizer and datasets ready.


## Trainers and training setups

We define **ImbalanceTrainer** (fixed class weights) and **DynamicWeightTrainer** (schedule: fixed / increasing / decreasing). Then run three setups.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    p_pos, r_pos, f1_pos, _ = precision_recall_fscore_support(
        labels, preds, labels=[1], average=None, zero_division=0
    )
    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        labels, preds, average="macro", zero_division=0
    )

    acc = (preds == labels).mean()

    return {
        "accuracy": float(acc),
        "precision_pos": float(p_pos[0]),
        "recall_pos": float(r_pos[0]),
        "f1_pos": float(f1_pos[0]),
        "f1_macro": float(f1_macro),
    }


class ImbalanceTrainer(Trainer):
    """Trainer with class-weighted CE loss only (no sampling)."""

    def __init__(self, class_weights=None, use_class_weights=True, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = class_weights
        self.use_class_weights = use_class_weights

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        if self.use_class_weights and self.class_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=self.class_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


class DynamicWeightTrainer(Trainer):
    """Trainer with dynamic class weights that can change per epoch (schedule: fixed / increasing / decreasing)."""

    def __init__(self, weight_schedule="fixed", base_weights=None, **kwargs):
        super().__init__(**kwargs)
        self.weight_schedule = weight_schedule
        self.base_weights = base_weights
        self.num_epochs = kwargs.get("args").num_train_epochs if kwargs.get("args") else EPOCHS

        if base_weights is not None:
            w0_base, w1_base = base_weights[0].item(), base_weights[1].item()
            if weight_schedule == "increasing":
                self.w1_start = w1_base * 0.5
                self.w1_end = w1_base
            elif weight_schedule == "decreasing":
                self.w1_start = w1_base * 1.5
                self.w1_end = w1_base
            else:
                self.w1_start = w1_base
                self.w1_end = w1_base
        else:
            self.w1_start = None
            self.w1_end = None

    def get_current_weights(self):
        if self.base_weights is None:
            return None
        if self.weight_schedule == "fixed":
            return self.base_weights
        if hasattr(self.state, "epoch") and self.state.epoch is not None:
            current_epoch = self.state.epoch - 1
        else:
            current_epoch = 0
        current_epoch = max(0, min(current_epoch, self.num_epochs - 1))
        progress = current_epoch / (self.num_epochs - 1) if self.num_epochs > 1 else 0.0
        w0 = self.base_weights[0].item()
        w1 = self.w1_start + progress * (self.w1_end - self.w1_start)
        return torch.tensor([w0, w1], dtype=torch.float32)

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")
        current_weights = self.get_current_weights()
        if current_weights is not None:
            loss_fct = nn.CrossEntropyLoss(weight=current_weights.to(model.device))
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        else:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, model.config.num_labels), labels.view(-1))
        return (loss, outputs) if return_outputs else loss


print("compute_metrics, ImbalanceTrainer, and DynamicWeightTrainer defined.")

Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Trainer initialized.


In [ ]:
# --- Setup 1: Original (fixed class weights, default linear LR) ---
print("=" * 60)
print("SETUP 1: Original (fixed weights, linear LR)")
print("=" * 60)

MODEL_DIR_1 = MODEL_BASE / "original"
MODEL_DIR_1.mkdir(parents=True, exist_ok=True)

model_1 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_1 = TrainingArguments(
    output_dir=str(MODEL_DIR_1 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_1 = ImbalanceTrainer(
    model=model_1,
    args=args_1,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    class_weights=class_weights if USE_CLASS_WEIGHTS else None,
    use_class_weights=USE_CLASS_WEIGHTS,
)
train_result_1 = trainer_1.train()
print("Setup 1 training finished.")

eval_metrics_1 = trainer_1.evaluate(dev_dataset)
print("\nDev metrics (Setup 1 - Original):")
for k, v in eval_metrics_1.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")

pred_output_1 = trainer_1.predict(dev_dataset)
dev_preds_1 = np.argmax(pred_output_1.predictions, axis=-1)
dev_labels = pred_output_1.label_ids
report_str_1 = classification_report(dev_labels, dev_preds_1, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (Setup 1):")
print(report_str_1)

dev_metrics_record_1 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_1.items()}
dev_metrics_record_1["classification_report"] = report_str_1
dev_metrics_record_1["setup"] = "original"
with open(OUTPUT_DIR / f"{OUTPUT_PREFIX}_original_dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_1, f, indent=2)

trainer_1.save_model(str(MODEL_DIR_1))
tokenizer.save_pretrained(str(MODEL_DIR_1))
print(f"Saved model to: {MODEL_DIR_1}")

Epoch,Training Loss,Validation Loss,Accuracy,Precision Pos,Recall Pos,F1 Pos,F1 Macro
1,0.506900,0.635919,0.563246,0.173121,0.962025,0.293436,0.488687
2,0.421500,0.476204,0.884248,0.428571,0.683544,0.526829,0.730444
3,0.360100,0.979816,0.912888,0.545455,0.455696,0.496552,0.724435
4,0.211800,0.957356,0.911695,0.526316,0.632911,0.574713,0.762723
5,0.096500,1.782654,0.909308,0.526316,0.379747,0.441176,0.695913
6,0.207700,1.708096,0.911695,0.535211,0.481013,0.506667,0.729087
7,0.013400,2.162454,0.906921,0.507246,0.443038,0.472973,0.710963


Training finished.


In [ ]:
# --- Setup 2: Decreasing class weights (default linear LR) ---
print("=" * 60)
print("SETUP 2: Decreasing class weights (linear LR)")
print("=" * 60)

MODEL_DIR_2 = MODEL_BASE / "decreasing_weights"
MODEL_DIR_2.mkdir(parents=True, exist_ok=True)

model_2 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_2 = TrainingArguments(
    output_dir=str(MODEL_DIR_2 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_2 = DynamicWeightTrainer(
    model=model_2,
    args=args_2,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    weight_schedule="decreasing",
    base_weights=base_class_weights,
)
w1_start = base_class_weights[1].item() * 1.5
w1_end = base_class_weights[1].item()
print(f"Weight schedule: w1 from {w1_start:.4f} -> {w1_end:.4f} (w0 fixed)")
train_result_2 = trainer_2.train()
print("Setup 2 training finished.")

eval_metrics_2 = trainer_2.evaluate(dev_dataset)
print("\nDev metrics (Setup 2 - Decreasing weights):")
for k, v in eval_metrics_2.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")

pred_output_2 = trainer_2.predict(dev_dataset)
dev_preds_2 = np.argmax(pred_output_2.predictions, axis=-1)
report_str_2 = classification_report(dev_labels, dev_preds_2, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (Setup 2):")
print(report_str_2)

dev_metrics_record_2 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_2.items()}
dev_metrics_record_2["classification_report"] = report_str_2
dev_metrics_record_2["setup"] = "decreasing_weights"
with open(OUTPUT_DIR / f"{OUTPUT_PREFIX}_decreasing_weights_dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_2, f, indent=2)

trainer_2.save_model(str(MODEL_DIR_2))
tokenizer.save_pretrained(str(MODEL_DIR_2))
print(f"Saved model to: {MODEL_DIR_2}")

Dev metrics:
eval_loss: 0.3124
eval_accuracy: 0.9236
eval_precision_pos: 0.5924
eval_recall_pos: 0.6281
eval_f1_pos: 0.6098
eval_f1_macro: 0.7837
eval_runtime: 6.9695
eval_samples_per_second: 300.4530
eval_steps_per_second: 9.4700
epoch: 7.0000

Classification report (dev):
              precision    recall  f1-score   support

      No PCL     0.9607    0.9546    0.9576      1895
         PCL     0.5924    0.6281    0.6098       199

    accuracy                         0.9236      2094
   macro avg     0.7766    0.7914    0.7837      2094
weighted avg     0.9257    0.9236    0.9246      2094

Saved dev metrics to: /vol/bitbucket/tq422/NLP_cw/output/roberta_pcl_imbalance_weights_only_dev_metrics.json
Saved dev predictions to: /vol/bitbucket/tq422/NLP_cw/output/dev.txt
Saved dev predictions to /vol/bitbucket/tq422/NLP_cw/dev.txt (root)


In [ ]:
# --- Setup 3: Decreasing class weights + cosine LR scheduler ---
print("=" * 60)
print("SETUP 3: Decreasing class weights + cosine LR scheduler")
print("=" * 60)

MODEL_DIR_3 = MODEL_BASE / "decreasing_weights_cosine_lr"
MODEL_DIR_3.mkdir(parents=True, exist_ok=True)

model_3 = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)
args_3 = TrainingArguments(
    output_dir=str(MODEL_DIR_3 / "checkpoints"),
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    num_train_epochs=EPOCHS,
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pos",
    greater_is_better=True,
    report_to="none",
    seed=SEED,
)

trainer_3 = DynamicWeightTrainer(
    model=model_3,
    args=args_3,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=EARLY_STOPPING_PATIENCE)],
    weight_schedule="decreasing",
    base_weights=base_class_weights,
)
print(f"Weight schedule: w1 from {w1_start:.4f} -> {w1_end:.4f}; LR scheduler: cosine")
train_result_3 = trainer_3.train()
print("Setup 3 training finished.")

eval_metrics_3 = trainer_3.evaluate(dev_dataset)
print("\nDev metrics (Setup 3 - Decreasing + cosine LR):")
for k, v in eval_metrics_3.items():
    if isinstance(v, (int, float)):
        print(f"  {k}: {v:.4f}")

pred_output_3 = trainer_3.predict(dev_dataset)
dev_preds_3 = np.argmax(pred_output_3.predictions, axis=-1)
report_str_3 = classification_report(dev_labels, dev_preds_3, target_names=["No PCL", "PCL"], digits=4, zero_division=0)
print("\nClassification report (Setup 3):")
print(report_str_3)

dev_metrics_record_3 = {k: (float(v) if isinstance(v, (np.floating, np.integer)) else v) for k, v in eval_metrics_3.items()}
dev_metrics_record_3["classification_report"] = report_str_3
dev_metrics_record_3["setup"] = "decreasing_weights_cosine_lr"
with open(OUTPUT_DIR / f"{OUTPUT_PREFIX}_decreasing_weights_cosine_lr_dev_metrics.json", "w", encoding="utf-8") as f:
    json.dump(dev_metrics_record_3, f, indent=2)

trainer_3.save_model(str(MODEL_DIR_3))
tokenizer.save_pretrained(str(MODEL_DIR_3))
print(f"Saved model to: {MODEL_DIR_3}")

Saved model and tokenizer to: /vol/bitbucket/tq422/NLP_cw/models/roberta_pcl_imbalance_weights_only


## Summary: compare all three setups

In [ ]:
# --- Summary: compare all three setups ---
print("=" * 60)
print("SUMMARY: Comparison of All Setups")
print("=" * 60)

results_summary = {
    "original": {
        "f1_pos": eval_metrics_1.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_1.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_1.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_1.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_1.get("eval_accuracy", 0),
    },
    "decreasing_weights": {
        "f1_pos": eval_metrics_2.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_2.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_2.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_2.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_2.get("eval_accuracy", 0),
    },
    "decreasing_weights_cosine_lr": {
        "f1_pos": eval_metrics_3.get("eval_f1_pos", 0),
        "f1_macro": eval_metrics_3.get("eval_f1_macro", 0),
        "precision_pos": eval_metrics_3.get("eval_precision_pos", 0),
        "recall_pos": eval_metrics_3.get("eval_recall_pos", 0),
        "accuracy": eval_metrics_3.get("eval_accuracy", 0),
    },
}

print(f"\n{'Metric':<20} {'Original':<12} {'Decreasing':<12} {'Decr+Cosine':<12}")
print("-" * 60)
for metric in ["f1_pos", "f1_macro", "precision_pos", "recall_pos", "accuracy"]:
    print(f"{metric:<20} {results_summary['original'][metric]:<12.4f} {results_summary['decreasing_weights'][metric]:<12.4f} {results_summary['decreasing_weights_cosine_lr'][metric]:<12.4f}")

summary_path = OUTPUT_DIR / f"{OUTPUT_PREFIX}_summary.json"
with open(summary_path, "w", encoding="utf-8") as f:
    json.dump(results_summary, f, indent=2)
print(f"\nSaved summary to: {summary_path}")

best_f1_pos = max(results_summary.items(), key=lambda x: x[1]["f1_pos"])
print(f"\nBest F1-Pos: {best_f1_pos[0]} ({best_f1_pos[1]['f1_pos']:.4f})")
best_f1_macro = max(results_summary.items(), key=lambda x: x[1]["f1_macro"])
print(f"Best F1-Macro: {best_f1_macro[0]} ({best_f1_macro[1]['f1_macro']:.4f})")

Saved test predictions to: /vol/bitbucket/tq422/NLP_cw/output/test.txt
Saved test predictions to /vol/bitbucket/tq422/NLP_cw/test.txt (root)


: 

In [ ]:
# Optional: produce test predictions using the best setup (by dev F1-Pos)
best_name = best_f1_pos[0]
best_trainer = trainer_1 if best_name == "original" else (trainer_2 if best_name == "decreasing_weights" else trainer_3)
print(f"Using best setup '{best_name}' for test predictions.")

test_df = pd.read_csv(
    RAW_DATA_DIR / "task4_test.tsv",
    sep="\t",
    names=["id", "par_id", "keyword", "country_code", "text"],
)
test_df["text"] = test_df["text"].astype(str).apply(strip_html_and_clean)
test_df["text"] = test_df["text"].replace("", " ")

test_dataset = PCLDataset(
    texts=test_df["text"],
    labels=np.zeros(len(test_df), dtype=int),
    tokenizer=tokenizer,
    max_length=MAX_LENGTH,
)
test_pred_output = best_trainer.predict(test_dataset)
test_preds = np.argmax(test_pred_output.predictions, axis=-1)

test_txt_path = OUTPUT_DIR / "test.txt"
with open(test_txt_path, "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to: {test_txt_path}")
with open(PROJECT_ROOT / "test.txt", "w", encoding="utf-8") as f:
    for p in test_preds:
        f.write(f"{p}\n")
print(f"Saved test predictions to {PROJECT_ROOT / 'test.txt'} (root)")